# LeetCode 3456 — Find Special Substring of Length K bằng Python

Link đề: https://leetcode.com/problems/find-special-substring-of-length-k/ · Độ khó: **Easy** · Chủ đề: String (chuỗi), hai con trỏ, nhóm ký tự liên tiếp (run)

Skeleton LeetCode:

```python
class Solution:
    def hasSpecialSubstring(self, s: str, k: int) -> bool:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu năm cách làm trên hàng nghìn chuỗi ngẫu nhiên). Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm xem trong `s` có **chuỗi con (substring) độ dài đúng `k`** thỏa ba điều kiện: (1) chỉ gồm **một loại ký tự** (như `"aaa"`), (2) ký tự **ngay trước** nó (nếu có) **khác** ký tự đó, (3) ký tự **ngay sau** nó (nếu có) cũng **khác** ký tự đó.
2. Nắm các công cụ Python: **cắt chuỗi** `s[i:i+k]`, kiểm tra "toàn một ký tự" bằng `set` hoặc `c * k`, **kiểm tra biên** (`i > 0`, `i + k < n`), **hai con trỏ** `l`, `r` trượt qua từng nhóm, `itertools.groupby`, và biểu thức chính quy `re` (chỉ để so sánh).
3. Đi từ cách **thử mọi vị trí bắt đầu `i`** → **nhóm ký tự liên tiếp (run) bằng hai con trỏ** → **`groupby`** → **biểu thức chính quy** → **đếm độ dài run trong một lượt**.
4. Hiểu mấu chốt: ba điều kiện của đề nghĩa là **chuỗi con đó chính là một _run cực đại_** (nhóm ký tự giống nhau liên tiếp, không thể kéo dài thêm ở hai phía) có **độ dài đúng `k`**. Vậy bài toán ⇔ "có run nào dài đúng `k` không?".
5. Tránh các lỗi kinh điển: dùng `>= k` thay vì `== k` (run dài hơn `k` **không** tính), quên kiểm tra ký tự **trước/sau**, `s[i - 1]` với `i = 0` (chỉ số âm quay vòng!), quên xét **run cuối cùng** khi kết thúc vòng lặp, và dùng `c * k in s` (không xét hàng xóm).

## 2. Đề bài & ví dụ

Cho chuỗi `s` và số nguyên `k`.

Hãy xác định xem có tồn tại một **chuỗi con (substring)** của `s` có độ dài **đúng `k`** thỏa mãn các điều kiện sau không:

1. Chuỗi con đó chỉ gồm **một ký tự duy nhất** (ví dụ `"aaa"` hoặc `"bbb"`).
2. Nếu có ký tự **ngay trước** chuỗi con thì nó phải **khác** ký tự của chuỗi con.
3. Nếu có ký tự **ngay sau** chuỗi con thì nó cũng phải **khác** ký tự của chuỗi con.

Trả về `true` nếu tồn tại chuỗi con như vậy, ngược lại trả về `false`.

**Ví dụ 1**

- Input: `s = "aaabaaa"`, `k = 3`
- Output: `true`
- Giải thích: chuỗi con `s[4..6] = "aaa"` thỏa mãn: độ dài `3`, toàn ký tự `a`, ký tự đứng trước là `b` (khác `a`), phía sau không có ký tự nào.

**Ví dụ 2**

- Input: `s = "abc"`, `k = 2`
- Output: `false`
- Giải thích: không có chuỗi con độ dài `2` nào gồm một ký tự duy nhất.

**Ràng buộc:** `1 <= k <= s.length <= 100`; `s` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
import random
import re
import string
from itertools import groupby

examples = [
    ("aaabaaa", 3, True),
    ("abc", 2, False),
]
for s, k, expected in examples:
    print(repr(s), "k =", k, "-> kỳ vọng", expected)

'aaabaaa' k = 3 -> kỳ vọng True
'abc' k = 2 -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Chuỗi con và cắt chuỗi `s[i:i+k]`

`s[a:b]` lấy các ký tự có chỉ số từ `a` đến `b - 1` (**không** gồm `b`). Vậy `s[i:i+k]` là chuỗi con **độ dài `k`** bắt đầu tại `i`. Nếu `i + k` vượt quá độ dài, Python **tự cắt bớt** (không báo lỗi) và chuỗi ngắn hơn `k` — vì thế ta chỉ xét `i` từ `0` đến `n - k`.

In [2]:
s = "aaabaaa"
k = 3
print(s[0:3], s[3:6], s[4:7])          # aaa baa aaa
print(len(s), "-> các vị trí bắt đầu hợp lệ: 0 ..", len(s) - k)
print(list(range(len(s) - k + 1)))     # [0, 1, 2, 3, 4]
print(s[5:9], "(bị cắt bớt, độ dài", len(s[5:9]), ")")

aaa baa aaa
7 -> các vị trí bắt đầu hợp lệ: 0 .. 4
[0, 1, 2, 3, 4]
aa (bị cắt bớt, độ dài 2 )


### 3.2. Kiểm tra "toàn một loại ký tự"

Có hai cách ngắn:

- `len(set(sub)) == 1`: `set` gom ký tự khác nhau; chỉ còn một phần tử nghĩa là mọi ký tự giống nhau.
- `sub == sub[0] * len(sub)`: nhân chuỗi lặp lại ký tự đầu rồi so sánh (`"a" * 3 = "aaa"`).

In [3]:
for sub in ["aaa", "aab", "b"]:
    print(sub, set(sub), len(set(sub)) == 1, sub == sub[0] * len(sub))
print("a" * 3, "ab" * 2)               # aaa abab

aaa {'a'} True True
aab {'a', 'b'} False False
b {'b'} True True
aaa abab


### 3.3. Ký tự ngay trước và ngay sau — kiểm tra biên

Với chuỗi con bắt đầu ở `i`, kết thúc ở `i + k - 1`:

- ký tự **ngay trước** là `s[i - 1]` và **chỉ tồn tại khi `i > 0`**;
- ký tự **ngay sau** là `s[i + k]` và **chỉ tồn tại khi `i + k < n`**.

Cẩn thận: ở Python `s[-1]` là ký tự **cuối** chuỗi; nếu bạn viết `s[i - 1]` khi `i = 0` thì **không có lỗi** nhưng lấy nhầm ký tự cuối.

In [4]:
s = "aaabaaa"
n, k = len(s), 3
for i in [0, 4]:
    before = s[i - 1] if i > 0 else None
    after = s[i + k] if i + k < n else None
    print(f"i={i}: chuỗi con={s[i:i+k]!r}, trước={before!r}, sau={after!r}")
print("s[-1] =", repr(s[-1]), "(ký tự cuối, KHÔNG phải 'không có ký tự trước')")

i=0: chuỗi con='aaa', trước=None, sau='b'
i=4: chuỗi con='aaa', trước='b', sau=None
s[-1] = 'a' (ký tự cuối, KHÔNG phải 'không có ký tự trước')


### 3.4. "Run" — nhóm ký tự giống nhau liên tiếp

Một **run** là đoạn **dài nhất** gồm các ký tự giống nhau đứng liền nhau. `"aaabaaa"` có các run: `aaa` (vị trí `0..2`), `b` (`3`), `aaa` (`4..6`). Một run **không thể kéo dài thêm** vì ký tự trước/sau nó đã khác. Đây chính là hình ảnh của điều kiện (2) và (3) trong đề.

In [5]:
s = "aaabaaa"
runs = []
i = 0
while i < len(s):
    j = i
    while j < len(s) and s[j] == s[i]:
        j += 1
    runs.append((s[i], j - i))          # (ký tự, độ dài run)
    i = j
print(runs)                              # [('a', 3), ('b', 1), ('a', 3)]

[('a', 3), ('b', 1), ('a', 3)]


### 3.5. Hai con trỏ `l` và `r`

Kỹ thuật **hai con trỏ**: `l` đứng ở đầu một run, `r` chạy sang phải **cho đến khi ký tự đổi** (`s[r] != s[l]`). Khi dừng, run là `s[l:r]` với độ dài `r - l`; sau đó đặt `l = r` để sang run kế tiếp. Mỗi ký tự được `r` đi qua đúng **một lần** ⇒ `O(n)`.

In [6]:
s = "aabbbc"
l, n = 0, len(s)
while l < n:
    r = l
    while r < n and s[r] == s[l]:
        r += 1
    print(f"run {s[l:r]!r}: l={l}, r={r}, độ dài {r - l}")
    l = r

run 'aa': l=0, r=2, độ dài 2
run 'bbb': l=2, r=5, độ dài 3
run 'c': l=5, r=6, độ dài 1


### 3.6. `itertools.groupby`

`groupby(s)` tự gom **các ký tự giống nhau liên tiếp** thành nhóm; mỗi nhóm trả về cặp `(ký_tự, iterator_các_phần_tử)`. Muốn độ dài nhóm ta dùng `len(list(group))` hoặc `sum(1 for _ in group)`. (Không giống `Counter`: `groupby` **không** gộp các nhóm cách nhau — `"aabaa"` cho `a`, `b`, `a` chứ không gộp hai nhóm `a`.)

In [7]:
print([(c, len(list(g))) for c, g in groupby("aaabaaa")])     # [('a', 3), ('b', 1), ('a', 3)]
print([(c, sum(1 for _ in g)) for c, g in groupby("aabbbc")]) # [('a', 2), ('b', 3), ('c', 1)]

[('a', 3), ('b', 1), ('a', 3)]
[('a', 2), ('b', 3), ('c', 1)]


### 3.7. Biểu thức chính quy `re` (chỉ để so sánh)

`re.finditer(r"(.)\1*", s)` tìm các **run**: `(.)` bắt một ký tự bất kỳ, `\1*` lặp lại **đúng ký tự vừa bắt** `0` hay nhiều lần. Mỗi kết quả `m.group()` là một run. Đây là cách ngắn nhưng khó đọc hơn với người mới; ta dùng như một cách đối chiếu.

In [8]:
print([m.group() for m in re.finditer(r"(.)\1*", "aaabaaa")])    # ['aaa', 'b', 'aaa']
print([m.group() for m in re.finditer(r"(.)\1*", "aabbbc")])     # ['aa', 'bbb', 'c']

['aaa', 'b', 'aaa']
['aa', 'bbb', 'c']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `s = "aaaab"`, `k = 3`. Thử từng vị trí bắt đầu `i = 0, 1, 2` (vì `n - k = 2`):

| `i` | chuỗi con `s[i:i+3]` | toàn một ký tự? | ký tự trước | ký tự sau | hợp lệ? |
|---|---|---|---|---|---|
| 0 | `aaa` | có | không có | `s[3] = a` (giống) | không |
| 1 | `aaa` | có | `s[0] = a` (giống) | `s[4] = b` | không |
| 2 | `aab` | không | — | — | không |

Đáp án `false`: tuy có chuỗi con `"aaa"` nhưng nó luôn dính thêm một `a` ở sát bên cạnh, vì **run `aaaa` dài `4`**, không phải `3`.

Còn với `s = "aaabaaa"`, `k = 3`: các run là `aaa`, `b`, `aaa` — có run dài **đúng `3`** nên đáp án `true`.

**Nhận xét quan trọng:** thay vì thử mọi `i` rồi kiểm tra ba điều kiện, ta chỉ cần liệt kê các **run** và hỏi "có run nào dài đúng `k` không?".

In [9]:
s, k = "aaaab", 3
n = len(s)
for i in range(n - k + 1):
    sub = s[i:i + k]
    same = len(set(sub)) == 1
    before = s[i - 1] if i > 0 else None
    after = s[i + k] if i + k < n else None
    ok = same and before != sub[0] and after != sub[0]
    print(f"i={i}: {sub!r}, toàn một ký tự={same}, trước={before!r}, sau={after!r} -> {ok}")
print("các run:", [(c, len(list(g))) for c, g in groupby(s)])

i=0: 'aaa', toàn một ký tự=True, trước=None, sau='a' -> False
i=1: 'aaa', toàn một ký tự=True, trước='a', sau='b' -> False
i=2: 'aab', toàn một ký tự=False, trước='a', sau=None -> False
các run: [('a', 4), ('b', 1)]


## 5. Cách 1 — Brute Force / baseline: thử mọi vị trí bắt đầu

**Ý tưởng.** Với mỗi `i` từ `0` đến `n - k`:

1. Lấy `sub = s[i:i+k]`; nếu `len(set(sub)) != 1` thì bỏ qua (`continue`).
2. `c = sub[0]`. Nếu `i > 0` và `s[i - 1] == c` thì bỏ qua (ký tự trước giống).
3. Nếu `i + k < n` và `s[i + k] == c` thì bỏ qua (ký tự sau giống).
4. Qua hết ba cửa ải ⇒ trả `True`. Hết vòng lặp mà chưa trả ⇒ `False`.

In [10]:
def has_special_brute(s: str, k: int) -> bool:
    n = len(s)
    for i in range(n - k + 1):                 # i: vị trí bắt đầu
        sub = s[i:i + k]
        if len(set(sub)) != 1:                 # điều kiện 1: toàn một ký tự
            continue
        c = sub[0]
        if i > 0 and s[i - 1] == c:            # điều kiện 2: ký tự trước (nếu có) phải khác
            continue
        if i + k < n and s[i + k] == c:        # điều kiện 3: ký tự sau (nếu có) phải khác
            continue
        return True
    return False


for s, k, expected in examples:
    print(has_special_brute(s, k), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp (Cách 1).** Có `n - k + 1` vị trí, mỗi vị trí cắt chuỗi và tạo `set` tốn `O(k)` ⇒ **Time `O(n · k)`**, **Space `O(k)`** (chuỗi con và `set`). Với `n <= 100` rất nhanh.

**Nút thắt:** ta kiểm tra lại cùng các ký tự nhiều lần (các chuỗi con chồng lên nhau). Hơn nữa phải nhớ ba điều kiện và hai kiểm tra biên. Nếu nhìn theo **run**, mỗi ký tự chỉ cần đọc **một lần**.

## 6. Ý tưởng thuật toán / bước đột phá

**Chuỗi con đặc biệt ⇔ một run cực đại có độ dài đúng `k`.** Vì sao?

- Chuỗi con toàn một ký tự `c` mà ký tự **trước** và **sau** (nếu có) đều khác `c` thì không thể mở rộng sang hai phía — nó **chính là** một run cực đại.
- Ngược lại, một run cực đại có độ dài `k` thỏa cả ba điều kiện.

Vậy ta chỉ cần **chia `s` thành các run** và kiểm tra có run nào có độ dài `== k` hay không.

Ví dụ `"aaabaaa"`, `k = 3` ⇒ run `aaa` (độ dài `3`) ⇒ `True`; `"aaaab"`, `k = 3` ⇒ run `aaaa` (độ dài `4`), `b` (độ dài `1`) ⇒ không có run dài `3` ⇒ `False`.

In [11]:
for s, k in [("aaabaaa", 3), ("aaaab", 3), ("abc", 2)]:
    runs = [(c, len(list(g))) for c, g in groupby(s)]
    lengths = [length for _, length in runs]
    print(f"s={s!r}, k={k}: runs={runs}, có run dài đúng {k}? -> {k in lengths}")

s='aaabaaa', k=3: runs=[('a', 3), ('b', 1), ('a', 3)], có run dài đúng 3? -> True
s='aaaab', k=3: runs=[('a', 4), ('b', 1)], có run dài đúng 3? -> False
s='abc', k=2: runs=[('a', 1), ('b', 1), ('c', 1)], có run dài đúng 2? -> False


## 7. Các cách tiếp theo

### Cách 2 — Hai con trỏ chia run (cách mình khuyên nộp)

**Giải thích từng đoạn.**

1. `l` là đầu của run hiện tại, bắt đầu `l = 0`.
2. `r = l`, rồi `while r < n and s[r] == s[l]: r += 1` đẩy `r` tới **đầu run kế tiếp** (hoặc hết chuỗi).
3. Độ dài run là `r - l`. Nếu `== k` ⇒ trả `True`.
4. `l = r` để sang run kế tiếp. Hết chuỗi (`l == n`) thì dừng ⇒ `False`.

In [12]:
def has_special_two_pointers(s: str, k: int) -> bool:
    n = len(s)
    l = 0
    while l < n:
        r = l
        while r < n and s[r] == s[l]:      # kéo r qua hết run bắt đầu ở l
            r += 1
        if r - l == k:                     # run cực đại có độ dài đúng k
            return True
        l = r                              # sang run kế tiếp
    return False


for s, k, expected in examples:
    print(has_special_two_pointers(s, k), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


In [13]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionTwoPointers:
    def hasSpecialSubstring(self, s: str, k: int) -> bool:
        l, n = 0, len(s)
        while l < n:
            r = l
            while r < n and s[r] == s[l]:
                r += 1
            if r - l == k:
                return True
            l = r
        return False


print(SolutionTwoPointers().hasSpecialSubstring("aaabaaa", 3))      # True

True


**Độ phức tạp Cách 2:** `r` chỉ tiến, không lùi, mỗi ký tự được đọc **một lần** ⇒ **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — `itertools.groupby`

**Giải thích.** `groupby(s)` đã chia sẵn các run. Với mỗi nhóm `g`, `sum(1 for _ in g)` là độ dài run. `any(... == k for ...)` trả `True` nếu có run nào dài `k`. Rất ngắn, đúng ý "có run dài `k` không?".

In [14]:
def has_special_groupby(s: str, k: int) -> bool:
    return any(sum(1 for _ in g) == k for _, g in groupby(s))


for s, k, expected in examples:
    print(has_special_groupby(s, k), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`** (không lưu nhóm; mỗi nhóm được đếm rồi bỏ).

### Cách 4 — Biểu thức chính quy `(.)\1*`

**Giải thích.** `re.finditer(r"(.)\1*", s)` duyệt các run (xem 3.7); `len(m.group())` là độ dài run. Cách này chỉ để đối chiếu — ngắn nhưng khó đọc với người mới, và chi phí cao hơn một chút do dùng `re`.

In [15]:
def has_special_regex(s: str, k: int) -> bool:
    return any(len(m.group()) == k for m in re.finditer(r"(.)\1*", s))


for s, k, expected in examples:
    print(has_special_regex(s, k), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`** (không tính bộ nhớ nội bộ của `re`).

### Cách 5 — Một lượt, đếm độ dài run hiện tại

**Giải thích từng đoạn.**

1. Biến `run` là độ dài run **đang đếm** (kết thúc ở ký tự hiện tại).
2. Duyệt `i, c`: nếu `i > 0` và `c == s[i - 1]` thì run **kéo dài** ⇒ `run += 1`.
3. Ngược lại là **bắt đầu run mới**: **trước khi** đặt `run = 1`, kiểm tra run **vừa kết thúc** (`run == k`?).
4. **Sau vòng lặp**, run cuối cùng chưa được kiểm tra ⇒ `return run == k`. Đây là chi tiết dễ quên!

In [16]:
def has_special_counter(s: str, k: int) -> bool:
    run = 0                                    # độ dài run đang đếm
    for i, c in enumerate(s):
        if i > 0 and c == s[i - 1]:
            run += 1                           # run kéo dài
        else:
            if run == k:                       # run vừa kết thúc (ký tự c khác ký tự trước)
                return True
            run = 1                            # bắt đầu run mới
    return run == k                            # kiểm tra run cuối cùng


for s, k, expected in examples:
    print(has_special_counter(s, k), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace Cách 2 (hai con trỏ) trên `s = "aaabaaa"`, `k = 3` và trên `s = "aaaab"`, `k = 3`; sau đó trace Cách 5 (đếm run) để thấy biến `run` thay đổi.

In [17]:
def trace_two_pointers(s, k):
    n = len(s)
    print(f"s = {s!r}, k = {k}, n = {n}")
    l = 0
    while l < n:
        r = l
        while r < n and s[r] == s[l]:
            r += 1
        print(f"  l={l}, r={r}: run {s[l:r]!r}, độ dài {r - l}", end="")
        if r - l == k:
            print(f" == k -> TRẢ True")
            return True
        print(f" != k -> sang run kế (l = {r})")
        l = r
    print("  hết chuỗi -> False")
    return False


assert trace_two_pointers("aaabaaa", 3) is True
assert trace_two_pointers("aaaab", 3) is False
assert trace_two_pointers("abc", 2) is False

s = 'aaabaaa', k = 3, n = 7
  l=0, r=3: run 'aaa', độ dài 3 == k -> TRẢ True
s = 'aaaab', k = 3, n = 5
  l=0, r=4: run 'aaaa', độ dài 4 != k -> sang run kế (l = 4)
  l=4, r=5: run 'b', độ dài 1 != k -> sang run kế (l = 5)
  hết chuỗi -> False
s = 'abc', k = 2, n = 3
  l=0, r=1: run 'a', độ dài 1 != k -> sang run kế (l = 1)
  l=1, r=2: run 'b', độ dài 1 != k -> sang run kế (l = 2)
  l=2, r=3: run 'c', độ dài 1 != k -> sang run kế (l = 3)
  hết chuỗi -> False


In [18]:
def trace_counter(s, k):
    print(f"s = {s!r}, k = {k}")
    run = 0
    for i, c in enumerate(s):
        if i > 0 and c == s[i - 1]:
            run += 1
            print(f"  i={i} c={c!r}: giống ký tự trước -> run = {run}")
        else:
            print(f"  i={i} c={c!r}: ký tự mới; run trước đó = {run}", end="")
            if run == k:
                print(" == k -> TRẢ True")
                return True
            print(" -> đặt run = 1")
            run = 1
    print(f"  hết chuỗi: run cuối = {run}, == k? {run == k}")
    return run == k


assert trace_counter("aaabaaa", 3) is True
assert trace_counter("aaaab", 3) is False
assert trace_counter("aab", 1) is True            # run 'b' ở cuối dài 1, được kiểm tra SAU vòng lặp

s = 'aaabaaa', k = 3
  i=0 c='a': ký tự mới; run trước đó = 0 -> đặt run = 1
  i=1 c='a': giống ký tự trước -> run = 2
  i=2 c='a': giống ký tự trước -> run = 3
  i=3 c='b': ký tự mới; run trước đó = 3 == k -> TRẢ True
s = 'aaaab', k = 3
  i=0 c='a': ký tự mới; run trước đó = 0 -> đặt run = 1
  i=1 c='a': giống ký tự trước -> run = 2
  i=2 c='a': giống ký tự trước -> run = 3
  i=3 c='a': giống ký tự trước -> run = 4
  i=4 c='b': ký tự mới; run trước đó = 4 -> đặt run = 1
  hết chuỗi: run cuối = 1, == k? False
s = 'aab', k = 1
  i=0 c='a': ký tự mới; run trước đó = 0 -> đặt run = 1
  i=1 c='a': giống ký tự trước -> run = 2
  i=2 c='b': ký tự mới; run trước đó = 2 -> đặt run = 1
  hết chuỗi: run cuối = 1, == k? True


Bảng trace Cách 2 với `s = "aaabaaa"`, `k = 3`:

| lần | `l` | `r` | run `s[l:r]` | `r - l` | `== 3`? | hành động |
|---|---|---|---|---|---|---|
| 1 | 0 | 3 | `aaa` | 3 | có | trả `True` |

(Với `s = "aaabaaa"` run đầu `aaa` đã đủ, kết thúc ngay.) Bảng với `s = "aaaab"`, `k = 3`:

| lần | `l` | `r` | run `s[l:r]` | `r - l` | `== 3`? | hành động |
|---|---|---|---|---|---|---|
| 1 | 0 | 4 | `aaaa` | 4 | không | `l = 4` |
| 2 | 4 | 5 | `b` | 1 | không | `l = 5` |
| — | 5 | — | hết chuỗi | — | — | trả `False` |

## 9. Code nộp LeetCode

Mình nộp Cách 2 (hai con trỏ): `O(n)` thời gian, `O(1)` bộ nhớ, dễ giải thích và đúng cả biên.

In [19]:
class Solution:
    def hasSpecialSubstring(self, s: str, k: int) -> bool:
        l, n = 0, len(s)
        while l < n:
            r = l
            while r < n and s[r] == s[l]:      # kéo r qua hết run bắt đầu ở l
                r += 1
            if r - l == k:                     # run cực đại có độ dài đúng k
                return True
            l = r                              # sang run kế tiếp
        return False


sol = Solution()
for s, k, expected in examples:
    print(repr(s), k, "->", sol.hasSpecialSubstring(s, k), "(kỳ vọng", expected, ")")

'aaabaaa' 3 -> True (kỳ vọng True )
'abc' 2 -> False (kỳ vọng False )


## 10. Vì sao thuật toán đúng

- **Các run chia nhỏ chuỗi:** vòng `while l < n` đi từ trái sang phải; mỗi lần dừng `r` ở **đầu run kế tiếp** (ký tự khác `s[l]`) hoặc ở `n`. Do đó `s[l:r]` là một run **cực đại**: ký tự trước `s[l - 1]` khác `s[l]` (vì nó thuộc run trước đó) và ký tự sau `s[r]` (nếu có) khác `s[l]`.
- **Run cực đại độ dài `k` thỏa đề:** chuỗi con `s[l:r]` toàn một ký tự, độ dài `k`, ký tự trước và sau (nếu có) khác nó ⇒ đúng ba điều kiện.
- **Chuỗi con thỏa đề phải là run cực đại:** nếu `s[i:i+k]` toàn ký tự `c` và hai hàng xóm (nếu có) khác `c` thì không thể kéo dài nó, nên đúng là một run cực đại, và nó sẽ được vòng lặp gặp với `r - l = k`.
- **Run dài hơn `k` không tính:** ví dụ `"aaaa"`, `k = 3`: mọi chuỗi con `"aaa"` đều dính thêm `a` ở sát bên ⇒ sai điều kiện (2) hoặc (3). Ta dùng `== k` chứ không `>= k` chính vì vậy.
- **Dừng đúng:** `l` luôn tăng ít nhất `1` mỗi vòng (`r > l` vì `s[l] == s[l]`), nên vòng lặp kết thúc sau tối đa `n` lần, và mỗi ký tự được `r` đi qua một lần ⇒ `O(n)`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi vị trí bắt đầu | `O(n · k)` | `O(k)` |
| 2. Hai con trỏ chia run | `O(n)` | `O(1)` |
| 3. `groupby` | `O(n)` | `O(1)` |
| 4. Biểu thức chính quy | `O(n)` | `O(1)` |
| 5. Đếm độ dài run một lượt | `O(n)` | `O(1)` |

Vì `k <= n <= 100`, Cách 1 cũng chỉ tối đa vài nghìn bước. Điểm đáng học là nhìn bài toán theo **run**, kỹ thuật dùng lại ở nhiều bài chuỗi (nén chuỗi, ký tự liên tiếp dài nhất...).

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng bằng `assert`.

In [20]:
sol = Solution()
impls = {
    "brute": has_special_brute,
    "two_pointers": has_special_two_pointers,
    "groupby": has_special_groupby,
    "regex": has_special_regex,
    "counter": has_special_counter,
    "leetcode": sol.hasSpecialSubstring,
}


def check_all(s, k, expected, label):
    for key, fn in impls.items():
        actual = fn(s, k)
        assert actual == expected, (label, key, s, k, actual, expected)
    print("PASS", label)


for idx, (s, k, expected) in enumerate(examples, 1):
    check_all(s, k, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Các chuỗi dưới đây là **tự dựng**, đáp án tính tay theo định nghĩa run (ghi trong nhãn).

In [21]:
check_all("a", 1, True, "một ký tự, k = 1: run 'a' dài 1")
check_all("aa", 1, False, "run 'aa' dài 2, không có run dài 1")
check_all("aa", 2, True, "run 'aa' dài 2 = k, k = n")
check_all("ab", 1, True, "hai run dài 1")
check_all("aaa", 2, False, "chỉ có run 'aaa' dài 3")
check_all("aaaa", 3, False, "run dài 4 > k: mọi 'aaa' đều dính thêm 'a'")
check_all("aaaab", 3, False, "run 'aaaa' dài 4, run 'b' dài 1")
check_all("baaab", 3, True, "run 'aaa' ở giữa, hai bên là 'b'")
check_all("aaab", 3, True, "run ở đầu chuỗi (không có ký tự trước)")
check_all("baaa", 3, True, "run ở cuối chuỗi (không có ký tự sau)")
check_all("aabbba", 2, True, "run 'aa' ở đầu; chú ý s[-1] = 'a' không được tính là 'ký tự trước'")
check_all("abcabc", 1, True, "mọi run dài 1")
check_all("abcabc", 2, False, "không có run dài 2")
check_all("a" * 100, 100, True, "n = 100, toàn ký tự, k = n")
check_all("a" * 100, 99, False, "run dài 100 != 99")
check_all("ab" * 50, 1, True, "xen kẽ, n = 100")
check_all("aabbbbcc", 3, False, "runs 2, 4, 2: không có run dài 3")
check_all("aabbbcc", 3, True, "runs 2, 3, 2: có run dài 3")

PASS một ký tự, k = 1: run 'a' dài 1
PASS run 'aa' dài 2, không có run dài 1
PASS run 'aa' dài 2 = k, k = n
PASS hai run dài 1
PASS chỉ có run 'aaa' dài 3
PASS run dài 4 > k: mọi 'aaa' đều dính thêm 'a'
PASS run 'aaaa' dài 4, run 'b' dài 1
PASS run 'aaa' ở giữa, hai bên là 'b'
PASS run ở đầu chuỗi (không có ký tự trước)
PASS run ở cuối chuỗi (không có ký tự sau)
PASS run 'aa' ở đầu; chú ý s[-1] = 'a' không được tính là 'ký tự trước'
PASS mọi run dài 1
PASS không có run dài 2
PASS n = 100, toàn ký tự, k = n
PASS run dài 100 != 99
PASS xen kẽ, n = 100
PASS runs 2, 4, 2: không có run dài 3
PASS runs 2, 3, 2: có run dài 3


### 12.2. Phản ví dụ cho các cách làm sai

Hai cách "trông có vẻ đúng": (a) dùng `>= k` (cho phép run dài hơn); (b) dùng `c * k in s` (bỏ qua điều kiện hàng xóm).

In [22]:
def wrong_at_least(s, k):
    return any(sum(1 for _ in g) >= k for _, g in groupby(s))     # SAI: run dài hơn k cũng được


def wrong_substring_in(s, k):
    return any(c * k in s for c in set(s))                          # SAI: không xét ký tự hai bên


print(wrong_at_least("aaaa", 3), "!=", has_special_groupby("aaaa", 3))
print(wrong_substring_in("aaaab", 3), "!=", has_special_groupby("aaaab", 3))
assert wrong_at_least("aaaa", 3) is True and has_special_groupby("aaaa", 3) is False
assert wrong_substring_in("aaaab", 3) is True and has_special_groupby("aaaab", 3) is False

True != False
True != False


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh chuỗi **đúng ràng buộc** (độ dài `1..100`, chữ thường, `1 <= k <= n`) theo hai kiểu: (1) ký tự ngẫu nhiên trên bảng chữ cái nhỏ (nhiều run ngắn); (2) ghép các **run có độ dài ngẫu nhiên** (nhiều run dài, hợp để thử `k` bằng/lệch độ dài run). Mọi cách phải khớp với Cách 1 (đúng theo định nghĩa của đề).

In [23]:
def valid(s, k):
    return 1 <= k <= len(s) <= 100 and all("a" <= c <= "z" for c in s)


def random_runs_string(rng, max_len=100):
    parts, total, prev = [], 0, ""
    while total < max_len:
        c = rng.choice([x for x in "abc" if x != prev])
        length = rng.randint(1, 6)
        if total + length > max_len:
            length = max_len - total
        parts.append(c * length)
        total += length
        prev = c
        if rng.random() < 0.15:
            break
    return "".join(parts)


rng = random.Random(3456)
count_true = count_false = 0
for trial in range(4000):
    if trial % 2 == 0:
        n = rng.randint(1, 100)
        alphabet = rng.choice(["a", "ab", "abc", string.ascii_lowercase])
        s = "".join(rng.choice(alphabet) for _ in range(n))
    else:
        s = random_runs_string(rng)
    k = rng.randint(1, len(s)) if rng.random() < 0.5 else rng.randint(1, min(len(s), 6))
    assert valid(s, k)
    ref = has_special_brute(s, k)
    for key, fn in impls.items():
        assert fn(s, k) == ref, (key, s, k)
    if ref:
        count_true += 1
    else:
        count_false += 1
print("PASS 4000 chuỗi ngẫu nhiên đúng ràng buộc; số ca True =", count_true, ", False =", count_false)
assert count_true > 100 and count_false > 100          # đảm bảo test có cả hai loại kết quả

PASS 4000 chuỗi ngẫu nhiên đúng ràng buộc; số ca True = 1392 , False = 2608


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `>= k` thay vì `== k`

In [24]:
def sai_lon_hon_bang(s, k):
    l, n = 0, len(s)
    while l < n:
        r = l
        while r < n and s[r] == s[l]:
            r += 1
        if r - l >= k:                      # SAI: run dài hơn k vẫn bị tính
            return True
        l = r
    return False


print("sai:", sai_lon_hon_bang("aaaa", 3), "| đúng:", Solution().hasSpecialSubstring("aaaa", 3))

sai: True | đúng: False


**Vì sao sai:** với `"aaaa"`, `k = 3`, mọi chuỗi con `"aaa"` đều có ký tự `a` sát bên (trước hoặc sau) nên vi phạm điều kiện (2)/(3). Chỉ run **dài đúng `k`** mới hợp lệ. **Cách sửa:** `r - l == k`.

### Lỗi 2: Quên kiểm tra ký tự trước / sau

In [25]:
def sai_quen_hang_xom(s, k):
    for i in range(len(s) - k + 1):
        if len(set(s[i:i + k])) == 1:        # SAI: chỉ kiểm tra toàn một ký tự, bỏ qua hai hàng xóm
            return True
    return False


print("sai:", sai_quen_hang_xom("aaaab", 3), "| đúng:", Solution().hasSpecialSubstring("aaaab", 3))

sai: True | đúng: False


**Vì sao sai:** `"aaa"` có trong `"aaaab"` nhưng ký tự sau nó là `a` (giống). Điều kiện (2) và (3) bị bỏ qua. **Cách sửa:** kiểm tra `s[i - 1]` (khi `i > 0`) và `s[i + k]` (khi `i + k < n`), hoặc dùng cách chia run.

### Lỗi 3: `s[i - 1]` khi `i = 0` — chỉ số âm quay vòng

In [26]:
def sai_chi_so_am(s, k):
    n = len(s)
    for i in range(n - k + 1):
        sub = s[i:i + k]
        if len(set(sub)) != 1:
            continue
        if s[i - 1] == sub[0]:               # SAI: i = 0 thì s[-1] là ký tự CUỐI chuỗi
            continue
        if i + k < n and s[i + k] == sub[0]:
            continue
        return True
    return False


print("sai:", sai_chi_so_am("aabbba", 2), "| đúng:", Solution().hasSpecialSubstring("aabbba", 2))
print("s[-1] của 'aabbba' là", repr("aabbba"[-1]))

sai: False | đúng: True
s[-1] của 'aabbba' là 'a'


**Vì sao sai:** ở `i = 0` không có ký tự phía trước, nhưng `s[-1]` lặng lẽ lấy ký tự **cuối** (`'a'`) rồi so sánh nhầm: run `"aa"` đầu chuỗi bị loại oan. **Cách sửa:** `i > 0 and s[i - 1] == c`.

### Lỗi 4: Quên run cuối cùng (cách đếm một lượt)

In [27]:
def sai_quen_run_cuoi(s, k):
    run = 0
    for i, c in enumerate(s):
        if i > 0 and c == s[i - 1]:
            run += 1
        else:
            if run == k:
                return True
            run = 1
    return False                              # SAI: chưa kiểm tra run cuối cùng


print("sai:", sai_quen_run_cuoi("aab", 1), "| đúng:", Solution().hasSpecialSubstring("aab", 1))

sai: False | đúng: True


**Vì sao sai:** run chỉ được kiểm tra **khi gặp một ký tự khác** kết thúc nó. Run **cuối cùng** không có ký tự sau để "đóng" nó nên không bao giờ được kiểm tra trong vòng lặp. Với `"aab"`, `k = 1`: run `b` dài `1` ở cuối chuỗi. **Cách sửa:** `return run == k` sau vòng lặp (hoặc dùng hai con trỏ / `groupby`, tự xử lý run cuối).

### Lỗi 5: Dùng `c * k in s` hoặc `s.count(c * k)`

In [28]:
def sai_chuoi_con(s, k):
    return any(c * k in s for c in set(s))        # SAI: 'aaa' in 'aaaa' là True


print("sai:", sai_chuoi_con("aaaa", 3), "| đúng:", Solution().hasSpecialSubstring("aaaa", 3))

sai: True | đúng: False


**Vì sao sai:** toán tử `in` chỉ hỏi "có xuất hiện không", không quan tâm ký tự hai bên. **Cách sửa:** dựa trên run (độ dài run `== k`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có "**chuỗi con toàn một ký tự, hai bên khác**" ⇒ đó là một **run cực đại**. Hãy nghĩ ngay tới **chia run** (hai con trỏ hoặc `groupby`).
- Khung hai con trỏ chia run: `l` ở đầu run; `r` chạy `while r < n and s[r] == s[l]`; xử lý `s[l:r]`; `l = r`.
- "Đúng `k`" ≠ "ít nhất `k`": `==` hay `>=` đổi hẳn đáp án.
- Python: `s[i - 1]` khi `i = 0` **không lỗi** mà lấy ký tự cuối. Luôn viết `i > 0 and ...` (hoặc `i + k < n and ...` ở phía phải).
- Kỹ thuật "đếm run trong một lượt": nhớ **kiểm tra run cuối** sau vòng lặp.
- `groupby` chỉ gom các phần tử **liền nhau** — đúng với khái niệm run.
- Tự kiểm tra bằng ba chuỗi: `"aaabaaa"` (có), `"aaaa"` với `k = 3` (không, run dài 4), `"aab"` với `k = 1` (có, run cuối).

## 15. Biến thể của bài

1. **Run dài nhất:** trả về độ dài của run dài nhất (liên quan **Consecutive Characters (1446)**).
2. **Đếm** số run có độ dài đúng `k`; hoặc **liệt kê** các vị trí bắt đầu của mọi chuỗi con đặc biệt.
3. **Ít nhất `k`:** có run dài `>= k` không (khi đó đơn giản hơn: chỉ cần `>=`).
4. **Nén chuỗi** (Run-Length Encoding): `"aaabaaa" -> "a3b1a3"` (liên quan **String Compression (443)**).
5. Bài cùng họ: **Longer Contiguous Segments of Ones than Zeros (1869)**, **Count and Say (38)**.

In [29]:
def special_starts(s, k):
    """Mọi vị trí bắt đầu của chuỗi con đặc biệt độ dài k."""
    starts, l, n = [], 0, len(s)
    while l < n:
        r = l
        while r < n and s[r] == s[l]:
            r += 1
        if r - l == k:
            starts.append(l)
        l = r
    return starts


def longest_run(s):
    return max(sum(1 for _ in g) for _, g in groupby(s))


def run_length_encode(s):
    return "".join(f"{c}{sum(1 for _ in g)}" for c, g in groupby(s))


def has_run_at_least(s, k):
    return any(sum(1 for _ in g) >= k for _, g in groupby(s))


print(special_starts("aaabaaa", 3), longest_run("aabbbc"), run_length_encode("aaabaaa"), has_run_at_least("aaaa", 3))
assert special_starts("aaabaaa", 3) == [0, 4]
assert special_starts("aaaab", 3) == []
assert longest_run("aabbbc") == 3
assert run_length_encode("aaabaaa") == "a3b1a3"
assert has_run_at_least("aaaa", 3) is True

[0, 4] 3 a3b1a3 True


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi vị trí bắt đầu | cắt `s[i:i+k]`, kiểm tra 3 điều kiện | `O(n · k)` | `O(k)` | bám sát đề, `n` nhỏ |
| 2. Hai con trỏ chia run | tìm run cực đại, so `r - l == k` | `O(n)` | `O(1)` | **nên nộp** |
| 3. `groupby` | `any(len(run) == k)` | `O(n)` | `O(1)` | ngắn gọn nhất |
| 4. Biểu thức chính quy | `(.)\1*` bắt từng run | `O(n)` | `O(1)` | để đối chiếu, không khuyến khích cho người mới |
| 5. Đếm run một lượt | `run += 1` / đặt lại, nhớ run cuối | `O(n)` | `O(1)` | khi muốn không dùng thư viện |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"aabb"[1:3]`, `len(set("aaa"))`, `"ab" * 2`, `Solution().hasSpecialSubstring("aaaa", 3)`, `Solution().hasSpecialSubstring("aabaa", 2)`.

**Bài 2 (tính tay).** Với `s = "aabbbbcc"` và `k = 4`: liệt kê các run (ký tự, độ dài) và cho biết đáp án; sau đó làm lại với `k = 3`.

**Bài 3 (điền TODO).** Hoàn thành `has_special_todo(s, k)` theo Cách 2 (hai con trỏ).

In [30]:
def has_special_todo(s, k):
    # TODO 1: l = 0, n = len(s)
    # TODO 2: while l < n: r = l, rồi kéo r qua hết run: while r < n and s[r] == s[l]: r += 1
    # TODO 3: nếu r - l == k thì return True
    # TODO 4: l = r ; cuối hàm return False
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `has_special_groupby2(s, k)` theo Cách 3 nhưng **không dùng `any`**: dùng vòng `for c, g in groupby(s)`, đếm độ dài nhóm bằng `len(list(g))`, và `return True` khi bằng `k`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_special(s, k)` đếm **có bao nhiêu** chuỗi con đặc biệt độ dài `k`. (b) Viết `longest_special(s)` trả về độ dài `k` **lớn nhất** sao cho tồn tại chuỗi con đặc biệt độ dài `k` (chính là độ dài run dài nhất).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"ab"`, `1`, `"abab"`, `False`, `True` (`"aabaa"` có run `aa` ở đầu và cuối). Bài 3: bốn dòng đúng theo các TODO: khởi tạo, vòng `while` ngoài, vòng `while` trong, so `== k`.
- **Mức 2 (vừa):** Bài 2: runs `(a,2) (b,4) (c,2)`; `k = 4` ⇒ `True` (run `b` dài `4`); `k = 3` ⇒ `False` (không run nào dài `3`, run `b` dài `4`). Bài 4: `for c, g in groupby(s): if len(list(g)) == k: return True`, cuối cùng `return False`.
- **Mức 3 (sâu):** Bài 5(a): cộng `1` cho mỗi run có `r - l == k` (không `return` sớm). 5(b): `max` độ dài các run; chú ý ví dụ `"aaaab"`: run dài nhất `4` nên `longest_special("aaaab") == 4`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert "aabb"[1:3] == "ab"
assert len(set("aaa")) == 1
assert "ab" * 2 == "abab"
assert Solution().hasSpecialSubstring("aaaa", 3) is False
assert Solution().hasSpecialSubstring("aabaa", 2) is True

# Bài 2
assert [(c, len(list(g))) for c, g in groupby("aabbbbcc")] == [("a", 2), ("b", 4), ("c", 2)]
assert Solution().hasSpecialSubstring("aabbbbcc", 4) is True
assert Solution().hasSpecialSubstring("aabbbbcc", 3) is False


# Bài 3
def has_special_todo_ans(s, k):
    l, n = 0, len(s)
    while l < n:
        r = l
        while r < n and s[r] == s[l]:
            r += 1
        if r - l == k:
            return True
        l = r
    return False


for s, k, expected in examples:
    assert has_special_todo_ans(s, k) == expected


# Bài 4
def has_special_groupby2(s, k):
    for c, g in groupby(s):
        if len(list(g)) == k:
            return True
    return False


for s, k, expected in examples:
    assert has_special_groupby2(s, k) == expected
assert has_special_groupby2("aaaa", 3) is False


# Bài 5(a)
def count_special(s, k):
    count, l, n = 0, 0, len(s)
    while l < n:
        r = l
        while r < n and s[r] == s[l]:
            r += 1
        if r - l == k:
            count += 1
        l = r
    return count


assert count_special("aaabaaa", 3) == 2
assert count_special("abc", 1) == 3
assert count_special("abc", 2) == 0
assert count_special("aaaa", 3) == 0


# Bài 5(b)
def longest_special(s):
    return max(len(list(g)) for _, g in groupby(s))


assert longest_special("aaaab") == 4
assert longest_special("abc") == 1
assert longest_special("aabbbc") == 3
# kiểm tra: độ dài lớn nhất k mà has_special trả True
for s in ["aaaab", "abc", "aabbbc", "aaabaaa"]:
    best = max(k for k in range(1, len(s) + 1) if Solution().hasSpecialSubstring(s, k))
    assert best == longest_special(s)
print("Tất cả đáp án bài tập đều đúng")

Tất cả đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (chia run bằng hai con trỏ; tìm run dài đúng `k`)

```python
class Solution:
    def hasSpecialSubstring(self, s: str, k: int) -> bool:
        l, n = 0, len(s)
        while l < n:
            r = l
            while r < n and s[r] == s[l]:
                r += 1
            if r - l == k:
                return True
            l = r
        return False
```

**Checklist**

- [ ] Hiểu "chuỗi con đặc biệt" = **run cực đại** độ dài **đúng** `k` (dùng `==`, không `>=`)?
- [ ] `r` dừng ở ký tự **khác** `s[l]` hoặc ở `n` (kiểm tra `r < n` **trước** `s[r]`)?
- [ ] Sau mỗi run, đặt `l = r` (để `l` luôn tăng, tránh vòng lặp vô hạn)?
- [ ] Nếu dùng cách đếm một lượt: đã kiểm tra **run cuối cùng** sau vòng lặp?
- [ ] Không dùng `s[i - 1]` khi `i = 0` (chỉ số âm quay vòng)?
- [ ] Đã thử `"aaabaaa"`, `k=3` (`True`), `"abc"`, `k=2` (`False`), `"aaaa"`, `k=3` (`False`), `"aab"`, `k=1` (`True`)?
- [ ] Skeleton đúng `class Solution` / `def hasSpecialSubstring(self, s: str, k: int) -> bool`?